# Sentinel Grid ML & Threat Analysis Notebook
#### Unsupervised Clustering & Anomaly Detection on Network and Honeypot Data
---
This notebook implements the first version of the machine learning threat analysis pipeline for Sentinel Grid. It serves as both a research prototype and a technical foundation that will later be converted into production ready Python modules for backend integration
This represents the first end-to-end implementation of our ML pipeline.

Its purpose is to:
- Process raw honeypot attack data
- Convert event-level logs into session-level behavioral features
- Analyze feature distributions
- Establish a baseline anomaly detection model
- Define and identify “bot-like” behavior

For my role as ML & Threat Analysis Engineer, I focus on:
- Extracting meaningful behavioral features from raw logs
- Detecting anomalous and automated behavior
- Identifying sessions that are “bot-like”
- Preparing ML outputs for backend consumption

Since honeypot logs don't include labeled ground truth, I decided on using an anomaly detection (unsupervised learning) approach. Anomaly detection identifies sessions that statistically deviate from normal system behavioral patterns. 

The assumption is:
- Most sessions represent baseline behavior
- Rare and statistically extreme sessions are more likely to be malicious
- We also have the option to add on clustering to categorize attack types.
---

### Pipeline
1. Load data  
2. Clean and parse timestamps  
3. Sessionize attacker activity  
4. Compute session-level features  
5. Plot feature distributions  
6. Train clustering/anomaly model  
7. Evaluate results  
8. Save outputs (CSV/JSON)

---

### Datasets Used

To design and validate the pipeline, I used two publicly available datasets.

- Structured Network Intrusion Dataset: 
I will first evaluate our clustering workflow on the CIC-IDS2017 intrusion dataset:  
https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset  
This dataset contains labeled network traffic including both normal activity and multiple cyberattacks (brute force, DoS, infiltration), making it a common benchmark for intrusion detection research and anomaly analysis. To create a human baseline, we use the BENIGN portion of this labeled network intrusion dataset by training the anomaly model on BENIGN flows only

- Zenodo CyberLab Honeynet Dataset:
Next, I will apply our pipeline to real honeypot data from:  
https://zenodo.org/records/3687527    
This dataset contains real attacker interaction logs (connect, auth attempts, commands, etc.) with timestamps and session_id and will be used to train and validate feature extraction on realistic attacker sessions. Used for attacker behvaior modeling.


Together, these datasets allow us to validate the workflow on structured intrusion data and then apply it to realistic honeypot logs, preparing the pipeline for future deployment on our team’s own SentinelGrid honeypot data.



In [12]:
#uncomment to check which python version/enviornment is running script 
#import sys
#print(sys.executable)

#imports
import os
import json
import gzip
import math
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score


### Shannon Entropy 
Shannon entropy measures password strength in bits, representing min number of guesses needed to guess it.

The Shannon entropy is defined as:
$$
H = -\sum_{i=1}^{n} p(x_i)\log_2 p(x_i)
$$

where:
- $p(x_i)$ is the probability of character $x_i$ in the password  
- $n$ is the number of unique characters  

This helps us measure how unpredictable a password string is.

- Higher entropy → more randomness → more human-like
- Medium entropy → common human style passwords
- Lower entropy → repetitive/simple → more bot-like

<u> Character distribution entropy:</u> Instead of estimating entropy from the allowed character pool, we compute it from the actual characters used. This better captures repetition patterns, structured guesses, and dictionary based attacks.

In our pipeline, entropy is computed per password attempt and can be aggregated per session (mean entropy or maximum entropy) to characterize attacker behavior.


In [ ]:
def shannon_entropy(s:str) ->float:
    #handles missing values or nons trings
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return 0.0
    
    s=str(s)
    if len(s) == 0:
        return 0.0
        
    #counts how many times each character appears
    #stored in dict - ex: "aab1"-> {'a':2, 'b':1, '1':1}
    counts= {}
    for ch in s:
        counts[ch]= counts.get(ch, 0)+1
    
    #compute probability by dividing count by total length of string
    #ex: [2,1,1]/4 -> [0.5, 0.25, 0.25]
    probs= np.array(list(counts.values()), dtype=float)/ len(s)
    #apply shannon entropy formula 
    #measures how unpredictable the character distribution is
    entropy= -(probs*np.log2(probs)).sum()

    #low values -> low entropy 
    #high values -> high entropy 
    return float(entropy)

### Network Intrusion Dataset

Help debug and verify preprocessing + ML steps before using honeypot logs. Only issue is that since there are no real human SSH command level SSH and password strings due to sensitive information.

In [ ]:
#path for network intrustion data
NETWORK_DIR = Path("../data/Kaggle CIC-IDS-2017")
csv_files = sorted(NETWORK_DIR.glob("*.pcap_ISCX.csv"))

#prints found CSV files
print("CSV files:", len(csv_files))
for f in csv_files[:5]:
    print(" -", f.name)

Found CSV files: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv


### Zenodo CyberLab Honeynet Dataset
Stand-in data for our future SentinelGrid data. Allows us to test the pipeline and train an unsupervised model on honeypot sessions to learn attacker patterns and behavioral structure. 